# 《PythAPCS123》單元 6-7：迴圈常見邏輯與控制變數陷阱排查

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**對應教材**：吳邦一老師《PythAPCS123：Python 程式設計從 APCS 實作 1 級到 3 級》第 17.1 節 for 迴圈注意事項（第 77 頁）

---

### 🌟 本單元學習導航地圖
歡迎來到【第六章 迴圈結構與控制流程】的第七個關鍵實戰單元！

在經歷了計數迴圈、累加器模式、while 條件迴圈、流程中斷跳轉、雙重巢狀迴圈與幾何星號排版之後，初學同學已經掌握了非常全面的迴圈火力。然而，在 APCS 考場與線上評判系統（OJ）的真實高壓環境下，有高達 80% 以上的初學者失分，並不是因為「不會寫迴圈」，而是踩進了**「極為隱蔽、直譯器不會報錯，但答案就是完全不對」的邏輯死結與陷阱**！
- 為什麼在 `for` 迴圈裡面手動寫 `i += 5`，下一輪 `i` 卻像被施了時光倒流魔法一樣，依然頑固地回到原本的數列？
- 為什麼區間範圍明明算了半天，最後總是莫名其妙「少跑一輪」或「多算一次」？
- 迴圈跑完之後，計數變數肚子裡殘留的數值到底是多少？
- 為什麼電腦的風扇突然瘋狂尖叫、畫面徹底卡死（無窮迴圈）？
- 多回合連續測試時，為什麼前一筆測資的答案會偷偷污染下一筆測資？

本單元將帶領各位程式冒險者化身為「頂尖除錯法醫」，專題剖析迴圈世界中最致命的七大考場隱形地雷，並提煉出一套實戰自檢 SOP，助你消滅 99% 的執行邏輯錯誤！

本單元精心規劃了八大循序漸進的除錯避坑微步進關卡：
1. **6-7-1 陷阱一：在 for 迴圈內手動修改計數變數無效的「自動校正機制」**
2. **6-7-2 陷阱二：左閉右開與負步進的差一邊界陷阱（Off-by-one Error）**
3. **6-7-3 陷阱三：迴圈結束後計數變數的「殘留值與生命週期陷阱」**
4. **6-7-4 陷阱四：while 迴圈計數器忘記更新導致 CPU 100% 的無窮死結**
5. **6-7-5 陷阱五：巢狀迴圈計數變數重名遮蔽（外層 i 內層又寫 i 互相踩腳）**
6. **6-7-6 陷阱六：累加器或旗標放錯位置（迴圈外未歸零導致多回合殘留污染）**
7. **6-7-7 陷阱七：走訪過程中指標亂跳引發的「跳格漏跑與陣列地毯抽換陷阱」**
8. **6-7-8 考場迴圈自檢 SOP：4 個關鍵問題消滅 99% 的迴圈執行錯誤**

> 💡 **學習小叮嚀（嚴格零提前依賴）**：本單元為第六章第七節，**嚴格禁止使用任何串列（List，`[...]`）、字典、集合或自訂函數（`def` / `return`）**！所有除錯案例皆使用純數值、字串走訪與純計數變數演示。請跟隨標準五步驟（**說明 ➔ 範例 ➔ 填空 ➔ 練習 ➔ 挑戰**），打造百毒不侵的迴圈底層功力！

### 🛡️ 6-7-1 陷阱一：在 for 迴圈內手動修改計數變數無效的「自動校正機制」

**生活故事：搭乘摩天輪的自動旋轉吊艙**  
請初學同學想像你去遊樂園搭乘大型摩天輪。  
摩天輪的輪軸由大型機械馬達帶動，吊艙一個接一個依照固定的機械節奏旋轉：第 1 號、第 2 號、第 3 號……  
如果你坐在 1 號吊艙裡，想要加快速度，於是在吊艙裡面拼命自己往前推、用力跳動，甚至在吊艙名牌上偷偷寫上「99 號」，這能改變摩天輪下一個轉過來的吊艙嗎？  
完全不可能！不管你在 1 號吊艙裡面如何折騰，時間一到，摩天輪機械馬達依然會無情且精確地把「第 2 號吊艙」送到轉軸定位點！你在吊艙內部的任何手動修改，全都是徒勞無功的！

**運作機制——for 迴圈的自動取值與強制覆蓋：**  
在 Python 中，`for i in range(5):` 的底層運作本質是：**每一輪迴圈剛開始時，Python 直譯器都會自動從 `range` 數列中抓出下一個值，並「強行覆蓋賦值」給變數 `i`！**  
請仔細觀察這段許多初學者寫過的驚悚程式碼：
```python
for i in range(5):
    print(f"進入回合，i = {i}")
    i = 100  # 初學者妄想把 i 變成 100，企圖讓迴圈立刻結束或跳步！
    print(f"手動改完後，i = {i}")
```
執行結果會令人大吃一驚：
- 第 0 輪：進入時 $i = 0$，手動改完後 $i = 100$；
- 但是到了第 1 輪剛開始，直譯器回過頭來，從 `range(5)` 中自動拿出了下一個數字 `1`，毫不留情地直接把 $i$ 覆蓋成了 `1`！
- 接著第 2 輪又是 `2`、第 3 輪又是 `3`……原本的手動修改在進入下一輪的瞬間被徹底抹殺！

**常見錯誤陷阱：**  
- **想在 for 迴圈內動態跳步**：例如遇到特殊情況想讓計數器「跳過 3 步」（寫了 `i += 3`），結果發現毫無作用，下一輪仍然只前進 1 步！
- **想提早終止卻手動竄改變數**：以為把計數變數改成大於上限的數值就能停止迴圈。記住：在 `for` 迴圈中，**唯一能提前停機的指令只有 `break`！唯一能跳步的指令只有 `continue`！**

**APCS 考試實務建議：**  
如果題目明確需要「依照條件動態跳步」（例如有時前進 1 步、有時遇到障礙要前進 3 步），**請果斷放棄 `for` 迴圈，改用 `while` 迴圈！** 在 `while` 迴圈中，變數的步進完全由你自己手動控制，才能真正實現靈活的跳躍步調！

In [ ]:
# [2] Code 範例區：揭密 for 迴圈內部手動修改變數的「失效真相」
print("=== 測試在 for 迴圈中手動竄改計數變數 i ===")

for i in range(5):
    print(f"--> [回合開始] 直譯器自動指派 i = {i}")
    # 企圖在內部手動竄改 i
    i = 99
    print(f"    [手動修改] 變數 i 被硬改成 {i}")
    # 觀察進入下一輪時，i 是否會聽從我們的修改？

print("=== 實驗結束！for 迴圈完全依照 range 預定軌道前進，不受內部竄改影響！ ===")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 某同學想要走訪數字 0 到 9，
# 但他希望只要一遇到數字 4，就立刻「真正停機」退出迴圈。
# 他原本錯誤地在迴圈內部寫了 i = 10，結果程式仍然跑完了 0 到 9！
# 請將錯誤的手動賦值替換為真正的終止關鍵字！
# ==========================================

last_seen = -1

for i in range(10):
    last_seen = i
    if i == 4:
        print(f"遇到關鍵數字 {i}，立刻執行正確的煞車！")
        # 請填入正確的中斷關鍵字（而非竄改 i）
        ___

print(f"成功煞車退出，最後記錄的數字為：{last_seen}（預期為 4）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 跳步前進的需求對決：for 的無奈 vs while 的靈活。
# 我們要模擬棋盤走子，棋盤長度為 10 格（座標 0 到 9）。
# 規則：正常情況下一步走 1 格，但如果在座標 3 踩到了「彈簧陷阱」，下一步必須一口氣向前跳躍 3 格（直接降落到座標 6）！
# 請撰寫正確的 while 迴圈實現動態跳步，
# 將每一輪實際停留的座標依序印出，並統計總共走了幾步 total_steps。
#
# 【公開測試資料 1】
# 步進路徑：0 -> 1 -> 2 -> 3 (彈簧跳躍+3) -> 6 -> 7 -> 8 -> 9
# 預期造訪座標數：total_steps = 8
#
# 【公開測試資料 2】
# 若彈簧設置在座標 2，踩到跳躍 4 格：
# 路徑：0 -> 1 -> 2 (+4) -> 6 -> 7 -> 8 -> 9
# 預期造訪座標數：total_steps = 7
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

pos = 0
total_steps = 0

while pos < 10:
    print(f"當前停留在座標：{pos}")
    total_steps += 1
    if pos == 3:
        print("  [踩到彈簧] 瞬間向前彈跳 3 格！")
        pos += 3
    else:
        pos += 1

print("抵達終點，總步數：", total_steps)

# 公開測資驗證：
assert total_steps == 8, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 動態能量損耗走訪（while 自由步進）：
# 機器人初始位於座標 x = 0，能量 energy = 12。
# 機器人前進規則：
# 1. 每次前進，若目前能量為偶數，向前邁進 2 格（x += 2），並消耗 2 點能量（energy -= 2）；
# 2. 若目前能量為奇數，向前邁進 1 格（x += 1），並消耗 1 點能量（energy -= 1）；
# 請使用 while 迴圈（條件為 energy > 0）模擬此過程，
# 計算當能量耗盡時，機器人最終抵達的座標位置 final_x，以及總共邁出了幾步 steps。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

x = 0
energy = 12
steps = 0

while energy > 0:
    steps += 1
    if energy % 2 == 0:
        x += 2
        energy -= 2
    else:
        x += 1
        energy -= 1

print(f"機器人最終位置：x = {x}，總共邁出 {steps} 步")
# 12 是偶數，每次減 2，會連續執行 6 次，每次走 2 格
# 最終 x = 6 * 2 = 12，步數 steps = 6
assert x == 12 and steps == 6, "挑戰題計算錯誤！"
print("挑戰題 6.7.1 驗證通過！")


### 🕳️ 6-7-2 陷阱二：左閉右開與負步進的差一邊界陷阱（Off-by-one Error）

**生活故事：鋸木頭的段數 vs 鋸切次數**  
請初學同學思考一個經典的數學謎題：  
如果要將一根 10 公尺長的木頭，每 1 公尺鋸成一段，總共可以鋸出 10 段木頭。  
請問：木工師傅總共需要動手「鋸幾次」？  
答案是 **9 次**，而不是 10 次！因為鋸了 9 刀之後，最後一段自然就分開了！  
又例如我們爬樓梯，從 1 樓爬到 5 樓，我們實際上只爬了「4 層樓的階梯」！  
在資訊科學的世界裡，這種因為「端點到底算不算進去」而引發的錯誤，擁有一個響噹噹的專有名詞——**「差一錯誤（Off-by-one Error，簡稱 OBOE）」**！它是程式界殺傷力最強大的隱形殺手！

**運作機制——Python range 永遠「含頭不含尾」：**  
Python 的 `range(start, stop, step)` 設計理念是嚴格的**「左閉右開區間 $[start, stop)$」**：
- 它會包含起點 $start$；
- 但是它**「絕對不會包含終點 $stop$」**！它只會跑到 $stop$ 的「前一步」就戛然而止！

**三大高危發病現場：**
1. **想包含 $N$ 卻忘記加 1**：  
   想計算 $1 + 2 + \dots + 10$，寫成了 `range(1, 10)`！結果只加到了 9，硬生生漏掉了最重要的 10，答案直接錯誤！正確寫法永遠是 `range(1, N + 1)`！
2. **負步進倒數想包含 1 卻寫成 0**：  
   想從 10 倒數到 1，寫成了 `range(10, 1, -1)`！請記住「含頭不含尾」，寫 1 就只會倒數到 2！想倒數到 1，終點必須寫成 **0**：`range(10, 0, -1)`！
3. **想倒數到 0 終點卻不知道怎麼寫**：  
   想從 5 倒數到 0（包含 0），終點必須再退一步寫成 **-1**：`range(5, -1, -1)`！

**常見錯誤陷阱：**  
- **負步進忘了第三參數**：寫成 `range(10, 0)`。因為預設步進是 $+1$，正步進從 10 永遠不可能走到 0，直譯器判定區間為空，迴圈一輪都不跑！

**APCS 考試實務建議（防差一黃金自檢法）：**  
在考場草稿紙上，永遠用**「最極端的超小測資（例如 $N = 1$ 或 $N = 2$）」**在腦海中代入！  
例如當 $N = 1$ 時，你的迴圈會跑幾次？如果是 `range(1, N)`，`range(1, 1)` 一次都不跑；如果是 `range(1, N + 1)`，`range(1, 2)` 剛好跑 1 次！透過極端值檢驗，差一錯誤無所遁形！

In [ ]:
# [2] Code 範例區：差一錯誤的兩大極端現場血淚對照
print("=== 慘劇現場一：想算 1 到 5 總和，卻漏了 5 ===")
wrong_sum = 0
for i in range(1, 5):  # 致命手滑：只跑到 4！
    wrong_sum += i
print(f"錯誤寫法 range(1, 5) 算出總和：{wrong_sum}（漏掉了 5！）")

correct_sum = 0
for i in range(1, 6):  # 正確寫法：range(1, N + 1)
    correct_sum += i
print(f"正確寫法 range(1, 6) 算出總和：{correct_sum}（正確！1+2+3+4+5=15）")

print()
print("=== 慘劇現場二：倒數火箭升空，想倒數到 0 該怎麼寫？ ===")
countdown_log = ""
for t in range(3, -1, -1):  # 終點寫 -1，才能真正跑到 0！
    print(f"倒數發射：T - {t}")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 體育老師要求同學進行 10 到 1 的折返跑倒數點名。
# 請補齊負步進 range 的三個參數，
# 確保數列精確包含：10, 9, 8, ..., 2, 1（剛好 10 個數字）！
# ==========================================

count_runs = 0

# 請填入起點、終點與負步進值
for val in range(___, ___, ___):
    count_runs += 1

print(f"倒數走訪次數：{count_runs} 次（必須精準等於 10 次）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 密閉整數區間偶數和：
# 給定閉區間 $[low, high]$，要求計算此區間內所有「偶數」的總和 even_sum。
# 請注意：端點 low 與 high 若為偶數，也必須被算進去！
# 請撰寫正確邊界的 for 迴圈，完成加總。
#
# 【公開測試資料 1】
# 設定：low = 4, high = 10
# 區間偶數：4, 6, 8, 10
# 預期：even_sum = 28
#
# 【公開測試資料 2】
# 設定：low = 3, high = 7
# 區間偶數：4, 6
# 預期：even_sum = 10
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

low = 4
high = 10
even_sum = 0

for num in range(low, high + 1):
    if num % 2 == 0:
        even_sum += num

print("區間偶數和：", even_sum)

# 公開測資驗證：
assert even_sum == 28, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 嚴格包含零的負步進位階轉換：
# 給定初始水溫 temp = 5 度。
# 冰庫降溫測試：溫度每輪下降 2 度，直到降至 -5 度以下為止。
# 也就是走訪範圍包括：5, 3, 1, -1, -3, -5。
# 請撰寫負步進的 for 迴圈（利用 range），
# 確保包含端點 5 與 -5，並統計所有非負溫度（temp >= 0）的個數 non_negative_count。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

non_negative_count = 0

# 從 5 降到 -5，每次減 2。終點必須寫 -6 或 -7 才能涵蓋 -5！
for t in range(5, -6, -2):
    print(f"量測溫度：{t} 度")
    if t >= 0:
        non_negative_count += 1

print(f"非負溫度（>= 0）次數：{non_negative_count} 次")
# 5度(非負), 3度(非負), 1度(非負), -1度, -3度, -5度
# 非負共 3 次
assert non_negative_count == 3, "挑戰題計算錯誤！"
print("挑戰題 6.7.2 驗證通過！")


### 👻 6-7-3 陷阱三：迴圈結束後計數變數的「殘留值與生命週期陷阱」

**生活故事：借用教室黑板的臨時草稿**  
想像學校舉辦數學競賽，借用一間空教室作為臨時考場。  
監考老師在黑板的角落隨手寫下計數：「第 1 號考生、第 2 號考生……第 10 號考生交卷」。  
考試結束鐘聲響起，全體考生離場。這時候，黑板角角落上那個數字「10」會自動人間蒸發嗎？  
不會！如果不拿板擦去擦，那個「10」就會一直幽靈般地留在黑板上！  
如果下一堂課的老師走進教室，誤把黑板上的「10」當作今天的作業頁數，整個教學進度就全亂了套！

**運作機制——Python 變數沒有迴圈區塊作用域：**  
在很多其他程式語言（如 C++ 或 Java）中，在迴圈開頭宣告的計數變數，只要一踏出迴圈，該變數就會立刻被系統銷毀（生命週期結束，外部無法存取）。  
**但是在 Python 中完全不是這樣！**  
在 Python 裡，`for i in range(5):` 中的變數 `i`，在迴圈結束後**「依然存活在記憶體中」**！  
那麼，此時 `i` 的值到底是多少？
- **正常走完的情況**：`i` 的值會停留在**「最後一輪所取到的數值」**（以 `range(5)` 為例，最後一輪取到 4，所以迴圈結束後 `i` 的殘留值是 **4**，而不是 5）！
- **中途 `break` 跳出的情況**：`i` 的值會停留在**「觸發 `break` 瞬間的那一個數值」**！
- **迴圈一次都沒跑的情況（空數列）**：如果迴圈事前 `range(0)` 一次都沒進去，而且之前從沒宣告過 `i`，這時在外部存取 `i` 就會當場引發致命的 **`NameError: name 'i' is not defined`**！

**常見錯誤陷阱：**  
- **誤以為迴圈結束後 $i$ 會等於 $stop$**：跑完 `range(1, 10)`，以為迴圈後 $i = 10$，拿去當除數或索引，結果實際上 $i$ 只有 9，引發差一錯誤。
- **依賴未跑迴圈的殘留值**：如果搜尋的目標剛好在第一輪前就因空迴圈結束，後面的程式碼誤讀了上一回合遺留下來的舊值，得出荒謬的假答案。

**APCS 考試實務建議：**  
如果需要在迴圈結束後使用找到的結果，**「絕不要直接依賴迴圈計數器」**！  
請在迴圈外事先宣告專屬的結果變數（例如 `ans = -1` 或 `target_idx = None`），在迴圈內明確賦值：`ans = i`，並配合 `break`。這樣既安全、意圖清晰，又能避免空迴圈導致的程式崩潰！

In [ ]:
# [2] Code 範例區：探測迴圈結束後，計數變數的真實殘留值
print("=== 實驗一：for i in range(1, 5) 正常走完全程 ===")
for i in range(1, 5):
    pass  # 正常跑動：i 依序為 1, 2, 3, 4

print(f"迴圈結束後，變數 i 的殘留值為：{i}（注意：是 4，而不是 5！）")

print()
print("=== 實驗二：在中途 break，觀察殘留值 ===")
for num in range(10, 20):
    if num == 13:
        print(f"在 num = {num} 觸發 break！")
        break

print(f"break 跳出後，變數 num 的殘留值為：{num}（停在觸發瞬間的 13）")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 搜尋 1 到 100 中第一個能被 37 整除的大於零整數。
# 為了避免直接依賴迴圈變數殘留值，
# 請在迴圈外事先宣告 result = -1，
# 並在找到目標時將其存入 result 並 break！
# ==========================================

# 1. 迴圈外安全宣告
result = -1

for candidate in range(1, 101):
    if candidate % 37 == 0:
        # 2. 明確賦值並煞車
        result = ___
        ___

print(f"搜尋結果：{result}（必須為 37）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 空迴圈防禦與預設值守護：
# 給定搜尋區間起點 start 與終點 end。
# 我們要在 range(start, end) 中尋找第一個大於 50 的數值。
# 若區間本身為空（例如 start >= end）或者完全沒找到，
# 答案 answer 必須安全維持預設值 -1，絕不能引發 NameError 或誤讀！
#
# 【公開測試資料 1】
# 設定：start = 40, end = 60
# 預期：找到第一個大於 50 的數值，answer = 51
#
# 【公開測試資料 2】
# 設定：start = 70, end = 60 (空區間，迴圈不跑)
# 預期：維持預設值 answer = -1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

start = 40
end = 60
answer = -1

for val in range(start, end):
    if val > 50:
        answer = val
        break

print("最終答案為：", answer)

# 公開測資驗證：
assert answer == 51, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 質數判定安全提取：
# 給定待檢查數字 n = 29。
# 我們要檢查 n 是否為質數：
# 在迴圈外設定 is_prime = True，
# 使用 for d in range(2, int(n**0.5) + 1):
# 若 n % d == 0，代表發現因數，將 is_prime 設為 False，
# 並用 factor_found 記錄下這個因數，然後 break！
# 若是質數，factor_found 必須安全保持為 0。
# 請計算 n = 29 時的 is_prime 與 factor_found。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

n = 29
is_prime = True
factor_found = 0

limit = int(n ** 0.5)
for d in range(2, limit + 1):
    if n % d == 0:
        is_prime = False
        factor_found = d
        break

print(f"檢查數字 {n}：是否為質數？{is_prime}，發現因數：{factor_found}")
assert is_prime is True and factor_found == 0, "挑戰題計算錯誤！"
print("挑戰題 6.7.3 驗證通過！")


### 🛑 6-7-4 陷阱四：while 迴圈計數器忘記更新導致 CPU 100% 的無窮死結

**生活故事：跑步機上的倉鼠與卡住的倒數計時器**  
想像一隻精力充沛的倉鼠跳進了滾輪裡跑步。  
如果滾輪連接著一個開關：「只要跑滿 100 圈，飼料箱就會打開並停止滾輪」。  
但是，如果計數滾輪的感應器接觸不良壞掉了，倉鼠每跑一圈，面板上的數字永遠停在「0 圈」！  
可憐的倉鼠就會在裡面沒日沒夜、永無止境地一直跑下去，直到體力耗盡、機器燒毀冒煙為止！  
在程式中，這就是惡名昭彰的**「無窮迴圈（Infinite Loop）」**！

**運作機制——while 迴圈三大要素的「致命斷腿」：**  
在單元 6-3 我們學過，任何一個健全的 `while` 迴圈都必須具備三大金剛要件：
1. **初始條件**（例如 `i = 0`）
2. **終止檢查**（例如 `while i < 10:`）
3. **步進更新**（例如 `i += 1`）

初學者最常見的世紀災難，就是**「忘記寫第 3 步（`i += 1`）」**！
```python
# 致命錯誤示範：電腦即刻進入狂暴狀態！
i = 0
while i < 5:
    print(i)
    # 悲劇發生：程式設計師手滑，忘記寫 i += 1！
```
追蹤一下直譯器的命運：
- 第一輪：$i = 0$，$0 < 5$ 成立，印出 0；
- 第二輪：$i$ 依然是 0！$0 < 5$ 依然成立，又印出 0；
- 第 1000 萬輪：$i$ 還是 0！條件永遠為真！
電腦的 CPU 使用率瞬間飆升到 100%，終端機被雪片般的 0 瘋狂洗版，瀏覽器分頁當場死當崩潰！

**更隱蔽的無窮死結——continue 跳步導致步進被略過：**
```python
# 隱蔽大坑：continue 搶在更新之前執行！
i = 0
while i < 5:
    if i == 2:
        continue  # 致命一擊：直接跳過下方的 i += 1！
    print(i)
    i += 1
```
當 $i = 2$ 時，`continue` 發動，下方的 `i += 1` 被狠狠略過！程式回到迴圈頂部，$i$ 依舊是 2，再次觸發 `continue`，陷入了看不見輸出的「寂靜死結」！

**APCS 考試實務建議（考場求生指南）：**  
1. **強制中斷快捷鍵**：若在本地終端機執行遇到程式卡死，請立刻猛按 **`Ctrl + C`** 發送中斷信號；若在 Google Colab，請點擊儲存格左側的「停止按鈕」或選單的「中斷執行階段」。
2. **更新步進放最前或用 for**：在 `while` 迴圈中使用 `continue` 前，必須先手動完成變數更新；若次數固定，優先選用自動推進的 `for` 迴圈！

In [ ]:
# [2] Code 範例區：展示安全更新的 while 迴圈 vs continue 防禦
print("=== 正確示範一：每輪確實推進計數器 ===")
count = 0
while count < 3:
    print(f"正常推進中：count = {count}")
    count += 1  # 絕不可忘記的靈魂指令！

print()
print("=== 正確示範二：while 搭配 continue 時的先更新守則 ===")
num = 0
while num < 5:
    # 守則：先推進步進，再做條件檢查與 continue！
    num += 1
    if num == 3:
        print("  [遇到 3] 執行 continue，跳過本輪印出！")
        continue
    print(f"處理數字：{num}")

print("=== 全部安全執行完畢，杜絕 CPU 100% 死結！ ===")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 以下 while 迴圈原本因為漏寫了更新步進，
# 導致變數 battery 永遠停在 10，陷入無窮迴圈！
# 請在迴圈體末尾補上電量每次遞減 2 的更新指令，
# 讓迴圈能夠在電量歸零時正常停止！
# ==========================================

battery = 10
cycles = 0

while battery > 0:
    print(f"設備運作中，剩餘電量：{battery}%")
    cycles += 1
    # 請補上電量每次扣除 2 的指令
    ___

print(f"電量耗盡，安全關機！總共運作了 {cycles} 個週期")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 對數折半縮小安全計數：
# 給定一正整數 N。
# 只要 N 大於 1，就將 N 整除以 2（N //= 2），並將折半次數 halves 累加 1。
# 請撰寫安全的 while 迴圈，確保每輪數值必定縮小，絕不陷入無窮死結。
#
# 【公開測試資料 1】
# 設定：N = 16
# 歷程：16 -> 8 -> 4 -> 2 -> 1
# 預期：折半次數 halves = 4
#
# 【公開測試資料 2】
# 設定：N = 25
# 歷程：25 -> 12 -> 6 -> 3 -> 1
# 預期：折半次數 halves = 4
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

N = 16
halves = 0

while N > 1:
    N //= 2
    halves += 1

print("總折半次數：", halves)

# 公開測資驗證：
assert halves == 4, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 考拉茲猜想（Collatz 3n + 1）安全步數計算：
# 給定初始正整數 n = 12。
# 規則：
# 1. 若 n 為偶數，則 n = n // 2；
# 2. 若 n 為奇數，則 n = 3 * n + 1；
# 只要 n 不等於 1，就持續進行上述變換，並將步數 step_count 累加 1。
# 請撰寫 while 迴圈完成此模擬，計算 n = 12 抵達 1 所需的總步數。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

n = 12
step_count = 0

while n != 1:
    if n % 2 == 0:
        n //= 2
    else:
        n = 3 * n + 1
    step_count += 1

print(f"考拉茲變換完成，抵達 1 的總步數：{step_count}")
# 12 -> 6 -> 3 -> 10 -> 5 -> 16 -> 8 -> 4 -> 2 -> 1 (共 9 步)
assert step_count == 9, "挑戰題計算錯誤！"
print("挑戰題 6.7.4 驗證通過！")


### 👥 6-7-5 陷阱五：巢狀迴圈計數變數重名遮蔽（外層 i 內層又寫 i 互相踩腳）

**生活故事：雙胞胎互相代打的荒謬話劇**  
學校話劇社排練一齣大戲，主角是一對長相一模一樣的雙胞胎兄弟。  
劇本原本安排：哥哥負責在外層的大客廳演整場戲（外層迴圈），弟弟只在廚房的暗室裡負責傳遞道具（內層迴圈）。  
結果正式演出時，導演偷懶，胸前名牌全寫著「男主角」！  
每當弟弟在廚房一登場，哥哥胸前的名牌就被強制抹黑；當弟弟演完下場時，哥哥已經徹底迷失了自己剛才到底講到第幾幕台詞，整齣大戲當場崩盤腰斬！

**運作機制——變數遮蔽引發的迴圈提前夭折：**  
在單元 6-5 我們強調過，巢狀迴圈嚴禁使用同名變數。  
但初學者在實際寫題或從別處複製貼上代碼時，這項錯誤依舊高居榜首！  
讓我們再次重溫這個恐怖的執行歷程：
```python
for i in range(3):       # 外層：原本打算跑 i = 0, 1, 2
    for i in range(3):   # 內層：手滑又用了變數 i！
        print(i, end=" ")
```
當外層啟動第一輪（$i = 0$）：
1. 進入內層迴圈，內層把 `i` 當作自己的變數，依序印出 `0 1 2`；
2. 當內層結束的那一瞬間，變數 `i` 肚子裡的殘留值停留在 **2**！
3. 回到外層迴圈，外層直譯器查看當前的 `i`：發現已經達到數列的末尾 2 了！外層直譯器誤以為整個任務已經跑完，直接拍拍屁股宣布結束！
原本預期 $3 	imes 3 = 9$ 次的輸出，**硬生生被閹割成只跑了 3 次！**

**最可怕的地方：直譯器「零報錯」！**  
這種錯誤不會產生任何 `SyntaxError`，直譯器會一路綠燈順暢執行到底，但計算出的數據量只剩下原本的九牛一毛！

**APCS 考試實務防禦規範：**  
- **肌肉記憶防禦**：
  - 二維網格/矩陣一律寫：`for r in range(R):` 搭配 `for c in range(C):`！
  - 數學純量枚舉一律寫：`for i in ...:` 搭配 `for j in ...:`，第三層寫 `for k in ...:`！
- **嚴格禁止複製貼上偷懶**：凡是複製迴圈結構，第一時間立刻將計數變數改成下一順位的字母！

In [ ]:
# [2] Code 範例區：抓出巢狀迴圈「同名互踩」的隱形殺手
print("=== 錯誤示範：外層 i 內層也寫 i（次數慘遭閹割）===")
bad_runs = 0
for i in range(3):
    for i in range(3):  # 致命重名遮蔽！
        bad_runs += 1
print(f"錯誤寫法實際執行次數：{bad_runs} 次（預期 9 次，慘遭縮水成 3 次！）")

print()
print("=== 正確示範：外層 r 內層 c（各自獨立，互不干擾）===")
good_runs = 0
for r in range(3):
    for c in range(3):  # 獨立名牌 c
        good_runs += 1
print(f"正確寫法實際執行次數：{good_runs} 次（完美達到 3 * 3 = 9 次！）")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 某同學原本要排版一個 3 x 4 的井字號矩陣，
# 但他不小心把內層變數也寫成了 row！
# 請將內層迴圈變數修正為 col，
# 確保排版出標準的 3 列 4 欄矩陣（總共 12 顆星）！
# ==========================================

total_stars = 0

for row in range(3):
    # 請將內層計數變數修正為 col
    for ___ in range(4):
        print("*", end="")
        total_stars += 1
    print()

print(f"印出星號總數：{total_stars} 顆（必須剛好為 12 顆）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 座標和矩陣正確計數：
# 給定列數 R = 3 與欄數 C = 3。
# 請使用雙重迴圈走訪每個格子 (r, c)（r, c 皆從 0 到 2）：
# 計算座標和 sum_rc = r + c。
# 若 sum_rc 是 2 的倍數，則將 even_cells 累加 1。
# 請務必使用獨立計數變數 r 與 c，絕不可同名遮蔽。
#
# 【公開測試資料 1】
# 設定：R = 3, C = 3
# 偶數和座標：(0,0)=0, (0,2)=2, (1,1)=2, (2,0)=2, (2,2)=4
# 預期：even_cells = 5
#
# 【公開測試資料 2】
# 設定：R = 2, C = 3
# 偶數和座標：(0,0)=0, (0,2)=2, (1,1)=2
# 預期：even_cells = 3
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

R = 3
C = 3
even_cells = 0

for r in range(R):
    for c in range(C):
        if (r + c) % 2 == 0:
            even_cells += 1

print("偶數和格子數：", even_cells)

# 公開測資驗證：
assert even_cells == 5, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 三重獨立變數立體體積走訪：
# 給定長方體三個維度大小：長 X = 2、寬 Y = 3、高 Z = 4。
# 我們要枚舉所有立體座標 (x, y, z)，其中：
# x 走訪 0 到 X-1，y 走訪 0 到 Y-1，z 走訪 0 到 Z-1。
# 檢查條件：若 (x + y + z) 能被 3 整除，則將 match_points 累加 1。
# 請使用 x, y, z 三個互不衝突的變數名稱完成三重迴圈！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

X = 2
Y = 3
Z = 4
match_points = 0

for x in range(X):
    for y in range(Y):
        for z in range(Z):
            if (x + y + z) % 3 == 0:
                match_points += 1

print(f"滿足條件的立體座標點數量：{match_points}")
# 總點數 2 * 3 * 4 = 24 點，每 3 個點就有 1 個餘數為 0，總計 8 點
assert match_points == 8, "挑戰題計算錯誤！"
print("挑戰題 6.7.5 驗證通過！")


### 🧹 6-7-6 陷阱六：累加器或旗標放錯位置（迴圈外未歸零導致多回合殘留污染）

**生活故事：沒倒掉昨天洗菜水的水槽**  
想像你在廚房洗菜。  
正確的SOP是：洗完第一批胡蘿蔔，把水槽的水排乾、沖洗乾淨（歸零重設），接著再放新水洗第二批青椒；  
但如果廚師太迷糊，洗完胡蘿蔔後「完全不換水」，直接把青椒丟進漂滿泥沙的舊水裡洗！  
結果第二批青椒不僅沒洗乾淨，反而沾滿了上一批留下來的泥巴，越洗越髒！  
在連續處理多筆資料時，這就是最致命的**「狀態未歸零之資料污染（State Pollution）」**！

**運作機制——外層迴圈每回合的「強制歸零」鐵則：**  
當題目要求「連續處理 $T$ 組測試資料」或「計算矩陣中每一列各自的總和」時：
- **每一列的累加器（`row_sum = 0`）必須放在哪裡？**  
  必須放在**「外層迴圈的內部、內層迴圈的正上方」**！  
  每一輪新列開始前，累加器都被強制倒空歸零，才能精確統計該列自己的數值！

看看這個慘烈的反面教材：
```python
# 致命錯誤示範：累加器放錯了樓層！
row_sum = 0  # 手滑大悲劇：放在了最外層外面！
for r in range(3):
    for c in range(3):
        row_sum += 1
    print(f"第 {r} 列總和：{row_sum}")
```
追蹤結果：
- 第 0 列算完：`row_sum` 是 3（看起來好像對）；
- 第 1 列算完：`row_sum` 竟然變成了 **6**（包含了第 0 列的舊資料！）；
- 第 2 列算完：`row_sum` 飆升到了 **9**！後面的回合全被前面的殘留值嚴重污染！

**布林旗標（Flag）的同等慘劇：**  
搜尋旗標也是一樣！如果旗標 `found = False` 放在最外層外面，一旦在第 1 回合被改成了 `True`，到了第 2 回合、第 3 回合，它永遠維持 `True`，後面的所有檢查全部失靈！

**APCS 考試實務建議：**  
在 APCS 考場上，評測系統通常會一口氣餵入多筆測試資料（Multi-testcases）。寫完程式後，請務必把視線移到外層迴圈頂端，確認：**「所有累加器、計數器、布林旗標，都在進入每一回合的第一時間完成歸零重設了嗎？」**

In [ ]:
# [2] Code 範例區：多回合累加器放錯位置的「污染慘劇」對比
print("=== 錯誤示範：累加器放在外層外面（上一回合殘留污染）===")
total_polluted = 0
for round_id in range(1, 4):
    for step in range(2):
        total_polluted += 10
    print(f"第 {round_id} 回合統計值：{total_polluted}（數值被上一回合污染了！）")

print()
print("=== 正確示範：每回合進入時強制歸零重設 ===")
for round_id in range(1, 4):
    round_clean = 0  # 關鍵時刻：每一輪開始前，確實歸零！
    for step in range(2):
        round_clean += 10
    print(f"第 {round_id} 回合統計值：{round_clean}（乾淨獨立，每回合都是 20！）")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 模擬統計 3 位學生的各自主修測驗總分（每位學生考 2 科）。
# 原程式誤將 student_total 放在了最外層，
# 導致後面的學生把前面學生的分數全部累加進去！
# 請將 student_total 的歸零語句移至外層迴圈內部！
# ==========================================

# 模擬 3 位學生
for s_id in range(1, 4):
    # 請在每位學生開始前，將個人總分歸零
    ___ = 0
    
    # 每人考 2 科，每科皆為 80 分
    for subject in range(2):
        score = 80
        student_total += score
        
    print(f"第 {s_id} 號學生個人總分：{student_total} 分（每位皆應為 160 分）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 每列偶數計數器獨立統計：
# 給定 3 列 3 欄的矩陣（列號 r 從 1 到 3，欄號 c 從 1 到 3）。
# 每一格的數值為 val = r * c。
# 請計算「每一列各自包含幾個偶數」：
# 每一列開始時，偶數計數器 row_even_count 必須歸零；
# 走訪該列的 3 個欄位，若為偶數則 row_even_count 累加 1；
# 該列結束時印出該列的偶數數量，
# 並將最後一列（r = 3）的偶數個數存入 last_row_even 進行驗證。
#
# 【公開測試資料 1】
# r=1: 1*1=1, 1*2=2, 1*3=3 -> 偶數 1 個
# r=2: 2*1=2, 2*2=4, 2*3=6 -> 偶數 3 個
# r=3: 3*1=3, 3*2=6, 3*3=9 -> 偶數 1 個
# 預期：last_row_even = 1
#
# 【公開測試資料 2】
# 若 r=2 列：偶數個數為 3
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

last_row_even = 0

for r in range(1, 4):
    row_even_count = 0  # 每一列獨立歸零
    for c in range(1, 4):
        if (r * c) % 2 == 0:
            row_even_count += 1
    print(f"第 {r} 列偶數數量：{row_even_count}")
    if r == 3:
        last_row_even = row_even_count

# 公開測資驗證：
assert last_row_even == 1, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 多回合質數檢驗旗標獨立歸零：
# 我們要依序檢驗三個數字 num_a = 9, num_b = 13, num_c = 15 是否為質數。
# 外層走訪 3 個回合 round_idx（1 到 3），分別設定當前檢查值 target：
# round_idx = 1 時 target = 9；
# round_idx = 2 時 target = 13；
# round_idx = 3 時 target = 15；
# 規則：每一回合開始前，布林旗標 is_prime 必須重設為 True！
# 內層使用 for d in range(2, target) 檢查因數，若 target % d == 0 則 is_prime = False 並 break。
# 統計 3 回合中，總共有幾個數字是質數 prime_total。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

prime_total = 0

for round_idx in range(1, 4):
    if round_idx == 1: target = 9
    elif round_idx == 2: target = 13
    else: target = 15
    
    # 關鍵時刻：每回合獨立重設旗標！
    is_prime = True
    for d in range(2, target):
        if target % d == 0:
            is_prime = False
            break
            
    if is_prime:
        print(f"回合 {round_idx}：數字 {target} 是質數！")
        prime_total += 1
    else:
        print(f"回合 {round_idx}：數字 {target} 是合數！")

print(f"質數總個數：{prime_total} 個（只有 13 是質數）")
assert prime_total == 1, "挑戰題計算錯誤！"
print("挑戰題 6.7.6 驗證通過！")


### 🪜 6-7-7 陷阱七：走訪過程中指標亂跳引發的「跳格漏跑與地毯抽換陷阱」

**生活故事：邊走路邊抽掉腳下的紅地毯**  
想像你在長長的走廊上邁步向前巡視房間。  
走廊的地板上鋪著一塊塊連續的紅地毯（編號 0, 1, 2, 3, 4）。  
當你正站在第 2 塊地毯上檢查時，如果後勤人員突然從你腳下一把抽掉了第 2 塊地毯，後面的所有地毯瞬間向前遞補一格！  
這時候，如果你毫不知情地「照常向前邁出一步」，你原本以為會踏上第 3 塊地毯，但因為地毯往前滑動了，你實際上踏上的已經是原本的「第 4 塊地毯」！原本的第 3 塊地毯被無情地跳過了！

**運作機制——索引位移（Index Shift）造成的跳格漏檢：**  
在後續章節學習容器處理（如刪除元素）時，初學者經常會犯一個致命錯誤：**「一邊用指標前進，一邊改變搜尋的基底」**！  
即使在我們當前純數值的迴圈中，這個陷阱也極其常見：
```python
# 典型錯誤：在 while 走訪中條件分支更新步調不一致！
idx = 0
while idx < 6:
    if idx % 2 == 0:
        idx += 2  # 偶數跳 2 格
    else:
        idx += 1  # 奇數跳 1 格
    # 如果判斷條件寫錯，有些格子就會被永遠跳過，導致漏檢！
```
當我們在搜尋特定的連續特徵時（例如尋找字串中的連續重複字元、數值跳躍）：
- 如果沒有精確維護指標的推進節奏；
- 或者在處理某個元素後，誤以為已經處理完畢而「多加了一次索引（Double Increment）」；
- 就會引發嚴重的「跳格漏跑」！

**防禦心法——只在必要時單一推進：**  
1. **單一推進原則**：一個回合之內，指標前進的職責必須高度統一，絕不允許在 `if` 裡面加一次、在外面又加一次！
2. **原地再次檢查（Don't Advance on Action）**：如果某一輪的處理導致後續資料向前遞補，指標當前位置就「絕不能前進」，必須留在原地對新遞補上來的資料重新進行一次檢查！

**APCS 考試實務建議：**  
在 APCS 歷屆真題中，例如字串連續重複字元消除、動態模擬遊戲中消除同色珠子，掌握「消掉後指標留在原地再次檢查」的嚴謹思維，是杜絕漏跑的最佳防禦守則！

In [ ]:
# [2] Code 範例區：展示雙重推進（Double Increment）導致的跳格漏跑
print("=== 錯誤示範：在 if 內部推進，外面又無腦推進（跳格漏檢）===")
pos = 0
visited_wrong = 0
while pos < 6:
    print(f"檢查位置 pos = {pos}")
    visited_wrong += 1
    if pos == 2:
        pos += 1  # 手滑：在裡面加了一次！
    pos += 1      # 外面又加了一次！導致位置 3 被硬生生跳過了！

print(f"錯誤走訪總位置數：{visited_wrong} 個（位置 3 慘遭遺漏！）")

print()
print("=== 正確示範：單一出口統籌推進 ===")
pos = 0
visited_correct = 0
while pos < 6:
    print(f"安全檢查位置 pos = {pos}")
    visited_correct += 1
    pos += 1  # 乾淨俐落，每輪必定且只推進 1 步

print(f"正確走訪總位置數：{visited_correct} 個（完整走訪 0 到 5！）")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 模擬質數因數連續除盡（以 2 為例）：
# 給定數值 n = 24，我們要計算 n 總共能被 2 整除幾次。
# 規則：只要 n 能被 2 整除（n % 2 == 0），
# 就將 n 縮小一半（n //= 2），並將計數器 div_count 累加 1；
# 當不能再被 2 整除時立刻停止！
# 請補齊 while 迴圈的條件式與縮小指令！
# ==========================================

n = 24
div_count = 0

# 只要 n 還能被 2 整除就持續進行
while ___ == 0:
    div_count += 1
    # 請將 n 縮小一半
    n = ___

print(f"24 總共能被 2 整除 {div_count} 次（預期為 3 次，最後 n = {n}）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 數字序列峰值計數（嚴防邊界超界與雙重步進）：
# 模擬檢查一段高度序列：
# 假設序列由公式 height = (i * 7) % 11 給定（i 從 0 到 9）。
# 依序計算每一點的高度：
# 若當前高度 height > 5，視為「高峰點」，peak_count 累加 1；
# 請撰寫迴圈完整走訪 i 從 0 到 9（共 10 個點），
# 確保每一點皆精準受檢，絕不跳格漏跑。
#
# 【公開測試資料 1】
# 公式產出 10 個高度：
# i=0: 0, i=1: 7, i=2: 3, i=3: 10, i=4: 6,
# i=5: 2, i=6: 9, i=7: 5, i=8: 1, i=9: 8
# 大於 5 的點：7, 10, 6, 9, 8
# 預期：peak_count = 5
#
# 【公開測試資料 2】
# 若只檢查前 5 個點（i 從 0 到 4）：
# 大於 5 的點：7, 10, 6
# 預期：peak_count = 3
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

peak_count = 0

for i in range(10):
    height = (i * 7) % 11
    if height > 5:
        peak_count += 1

print("高峰點總數：", peak_count)

# 公開測資驗證：
assert peak_count == 5, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 連續減法模擬輾轉相除取餘數：
# 給定被除數 a = 38 與除數 b = 7。
# 在不直接使用 % 運算子的前提下，
# 使用 while 迴圈模擬連續扣除：
# 只要 a >= b，就讓 a -= b，並將減法次數 sub_times 累加 1。
# 迴圈結束後，a 的剩餘值即為餘數 remainder！
# 請計算 38 除以 7 的商數 sub_times 與餘數 a。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

a = 38
b = 7
sub_times = 0

while a >= b:
    a -= b
    sub_times += 1

remainder = a
print(f"商數：{sub_times}，餘數：{remainder}（驗證：38 = 7 * {sub_times} + {remainder}）")
assert sub_times == 5 and remainder == 3, "挑戰題計算錯誤！"
print("挑戰題 6.7.7 驗證通過！")


### 📋 6-7-8 考場迴圈自檢 SOP：4 個關鍵問題消滅 99% 的迴圈執行錯誤

**生活故事：民航機起飛前的飛行員檢查清單（Checklist）**  
現代民航飛機擁有世界上最複雜的機械與電子儀表系統。  
但是在每一次起飛之前，無論機長飛行過一萬小時還是兩萬小時，兩位飛行員都必須拿出一張印滿項目的**「飛行前標準檢查清單（Before Takeoff Checklist）」**：  
副機長念出一項：「襟翼（Flaps）？」，機長確認並複誦：「襟翼設為 5 度，已確認！」  
副機長念下一項：「自動煞車？」，機長確認：「已設為 RTO，確認！」  
正是這套嚴謹死板、絕不憑記憶僥倖的 SOP 清單，守護了全球每天數萬架次班機的安全起降！

**運作機制——APCS 考場迴圈終極除錯 4 問：**  
當你在考場上寫完一段迴圈程式碼，按下滑鼠送出評判（Submit）之前，請強制自己花費 30 秒，依序盤點以下 4 個致命問題：

1. **【第一問：邊界有包含端點嗎？】（Check Boundary）**  
   - 我的 `range` 是不是寫成了 `range(1, N)`？（漏掉了 $N$！）  
   - 負步進倒數是不是寫成了 `range(N, 1, -1)`？（漏掉了 1！）  
   - `while` 條件式到底是 `>=` 還是 `>`？端點等於時到底要不要進去？
2. **【第二問：每回合的狀態有歸零嗎？】（Check Reset）**  
   - 累加器（`total = 0`）、計數器（`count = 0`）、布林旗標（`found = False`）是不是放在最外層外面？  
   - 當進入第 2 筆、第 3 筆測試資料時，它們會不會帶著上一輪的髒水繼續跑？
3. **【第三問：巢狀迴圈的變數名稱有撞車嗎？】（Check Naming）**  
   - 外層是 `r`，內層有沒有手滑又寫成 `r`？  
   - 外層是 `i`，內層是不是獨立的 `j`？
4. **【第四問：while 迴圈保證步進且必能終止嗎？】（Check Termination）**  
   - 變數是否有確實更新（`i += 1`）？  
   - `continue` 是不是搶在更新步進之前執行了？  
   - 是否存在一種極端情況，導致條件永遠為真、電腦死當？

**常見錯誤陷阱：**  
憑感覺「改改看」。很多同學一旦測試不過，就盲目在代碼裡加個 1 或減個 1，結果越改洞越大。請嚴格依照 4 問 SOP 逐一理性排查！

**APCS 考試實務建議：**  
只要在考場落實這 4 問 SOP，你就能在按下提交鍵之前，主動消滅 99% 的非預期 Wrong Answer 與 Time Limit Exceeded，穩穩奪下滿分！

In [ ]:
# [2] Code 範例區：APCS 考場迴圈自檢 SOP 綜合實戰排查
print("=== 模擬考場送出前自檢：計算 1 到 N 中所有 3 的倍數和 ===")

N = 10
# 檢核 1：邊界確認（必須到 N+1 才能包含 N）
# 檢核 2：初始歸零（sum_multiples = 0）
sum_multiples = 0

for i in range(1, N + 1):
    if i % 3 == 0:
        sum_multiples += i

print(f"1 到 {N} 中 3 的倍數和：{sum_multiples}（3 + 6 + 9 = 18）")
assert sum_multiples == 18, "自檢未通過！"
print("=== 4 問自檢全部通過，程式碼穩如泰山！ ===")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 請運用「考場 4 問 SOP」修正以下代碼：
# 計算多組測資每組的奇數個數。
# 代碼存在兩個致命 Bug：
# 1. odd_count 未在每組開始時歸零；
# 2. range 漏加 1 導致最大值未受檢。
# 請補齊每輪歸零與正確的 range 邊界！
# ==========================================

# 模擬 2 組測資：每組檢查 1 到 5 的數字
for group in range(1, 3):
    # 修正 1：每組開始前必須歸零！
    odd_count = ___
    
    # 修正 2：必須跑到 5+1 才能包含 5！
    for val in range(1, ___):
        if val % 2 == 1:
            odd_count += 1
            
    print(f"第 {group} 組奇數個數：{odd_count}（必須為 3 個：1, 3, 5）")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 綜合檢核實戰：連續整數區間偶數最大值查找。
# 給定區間 $[A, B]$（包含 A 與 B）。
# 尋找該區間內「最大的偶數」max_even。
# 若區間內完全沒有偶數，維持預設值 -1。
# 請運用「自檢 SOP」確保：
# 1. 包含端點 B（range 寫法）；
# 2. 初始值安全設定為 -1；
# 3. 倒序搜尋加速（從 B 倒數回 A，找到第一個偶數立刻 break！）。
#
# 【公開測試資料 1】
# 設定：A = 11, B = 19
# 偶數包含：12, 14, 16, 18
# 預期：max_even = 18
#
# 【公開測試資料 2】
# 設定：A = 11, B = 11 (單一奇數)
# 預期：max_even = -1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

A = 11
B = 19
max_even = -1

# 倒序搜尋：從 B 倒數到 A
for num in range(B, A - 1, -1):
    if num % 2 == 0:
        max_even = num
        break

print("區間最大偶數：", max_even)

# 公開測資驗證：
assert max_even == 18, "公開測資 1 驗證失敗"
print("公開測資 1 通過！")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 考場實戰無死角檢核：二維方陣特定數字出現頻率統計。
# 給定大小 N = 4 的方陣。
# 每個格子 (r, c)（r 從 1 到 N，c 從 1 到 N）的值為 val = (r * c) % 5。
# 請精確統計數值「0」在整個方陣中出現的總次數 zero_freq。
# 自檢要求：
# 1. 外層 r 內層 c 絕不重名；
# 2. 邊界完整涵蓋 1 到 N；
# 3. 計數器 zero_freq 正確累加。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

N = 4
zero_freq = 0

for r in range(1, N + 1):
    for c in range(1, N + 1):
        val = (r * c) % 5
        if val == 0:
            zero_freq += 1

print(f"方陣中 0 出現的總次數：{zero_freq} 次")
# 1~4 中沒有 5 的倍數，所以 r*c 不可能被 5 整除（5是質數）
# 因此 (r * c) % 5 永遠不會是 0！總次數必為 0！
assert zero_freq == 0, "挑戰題計算錯誤！"
print("挑戰題 6.7.8 驗證通過！")


## 🏁 單元 6-7 總結與 APCS 解題心法回顧

恭喜各位學習者！你已經順利完成了這堂含金量最高的「考場防坑大特訓」！讓我們重溫這七大陷阱與自檢 SOP：

1. **for 迴圈內部手動修改無效**：
   - 每輪開始直譯器強制覆蓋取值；需要動態跳步請改用 `while`，提前結束請用 `break`。
2. **差一邊界陷阱（Off-by-one）**：
   - 牢記 `range` 左閉右開含頭不含尾；想包含 $N$ 必寫 $N+1$，負步進倒數到 1 必寫 0。
3. **迴圈計數器殘留值**：
   - 迴圈結束後計數變數依然存活，其值為最後一輪取到之值；對外輸出務必使用獨立專用變數。
4. **while 漏寫步進之無窮死結**：
   - 務必確保迴圈三要素完備；使用 `continue` 時切記步進更新不可被略過。
5. **巢狀迴圈同名遮蔽**：
   - 嚴禁內外層共用變數名稱；銘記橫列寫 `r`、直欄寫 `c`、數學寫 `i, j, k`。
6. **狀態未歸零之資料污染**：
   - 每一回合開始的第一時間，所有累加器、計數器與布林旗標必須在外層內部強制歸零。
7. **指標亂跳與跳格漏檢**：
   - 保持單一推進原則，若資料向前遞補則指標留原地重新檢查，嚴防雙重遞增。
8. **考場迴圈自檢 4 問 SOP**：
   - 邊界有包含嗎？狀態有歸零嗎？變數有撞車嗎？while 保證終止嗎？

在下一個單元 **單元 6-8：APCS 考場迴圈輸入實戰模式：固定筆數、哨兵終止與未知行數（EOF）處理** 中，我們將直奔競賽實戰，全盤解密線上評測系統的三大輸入架構，為第六章劃下最完美的句點！